In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

In [2]:
# Load the OECD tourism data into a data frame.
# We’re interested in the following columns:
# – LOCATION — A three-letter abbreviation for the country name
# – SUBJECT — Either INT_REC (for tourist funds received) or INT-EXP (for tourist expenses). 
# – TIME — A year (integer)
# – Value — A float indicating thousands of dollars
df = pd.read_csv(
    '../../../data/oecd_tourism.csv',
    engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    usecols = ['LOCATION', 'SUBJECT', 'TIME', 'Value']
)

In [3]:
df

,LOCATION,SUBJECT,TIME,Value
0,AUS,INT_REC,2008,31159.8
1,AUS,INT_REC,2009,29980.7
2,AUS,INT_REC,2010,35165.5
3,AUS,INT_REC,2011,38710.1
4,AUS,INT_REC,2012,38003.7
...,...,...,...,...
1229,SRB,INT-EXP,2015,1253.644
1230,SRB,INT-EXP,2016,1351.098
1231,SRB,INT-EXP,2017,1549.183
1232,SRB,INT-EXP,2018,1837.317


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1234 entries, 0 to 1233
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   LOCATION  1234 non-null   string[pyarrow]
 1   SUBJECT   1234 non-null   string[pyarrow]
 2   TIME      1234 non-null   int64[pyarrow] 
 3   Value     1234 non-null   double[pyarrow]
dtypes: double[pyarrow](1), int64[pyarrow](1), string[pyarrow](2)
memory usage: 41.1 KB


In [5]:
df['TIME'] = df['TIME'].astype('int16')
# A small amount of memory can be saved by optimizing how year values are stored.

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1234 entries, 0 to 1233
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   LOCATION  1234 non-null   string[pyarrow]
 1   SUBJECT   1234 non-null   string[pyarrow]
 2   TIME      1234 non-null   int16          
 3   Value     1234 non-null   double[pyarrow]
dtypes: double[pyarrow](1), int16(1), string[pyarrow](2)
memory usage: 33.9 KB


In [6]:
# Find the five countries that received the greatest amount of tourist dollars, on
# average, across years in the data set.

df.loc[df['SUBJECT'] == 'INT_REC'].groupby('LOCATION')['Value'].mean().sort_values().round(1).iloc[-5:]

LOCATION
GBR     51752.1
DEU     53408.6
FRA     65063.3
ESP     69655.8
USA    201613.5
Name: Value, dtype: double[pyarrow]

In [7]:
# Find the five countries whose citizens spent the least amount of tourist dollars,
# on average, across years in the data set.

df.loc[df['SUBJECT'] == 'INT-EXP'].groupby('LOCATION')['Value'].mean().sort_values().round(1).iloc[:5]

LOCATION
MLT     387.8
CRI     867.1
LVA     919.5
ISL    1072.8
HRV    1115.6
Name: Value, dtype: double[pyarrow]

In [8]:
df_oecd_locations = pd.read_csv(
    '../../../data/oecd_locations.csv',
    engine = 'pyarrow',
    dtype_backend = 'pyarrow',
    names = ['location', 'location_full_name']
)

In [9]:
df_oecd_locations

,location,location_full_name
0,AUS,Australia
1,AUT,Austria
2,BEL,Belgium
3,CAN,Canada
4,DNK,Denmark
5,FIN,Finland
6,FRA,France
7,DEU,Germany
8,HUN,Hungary
9,ITA,Italy


In [10]:
# Join these two data frames together into a new one. In the new data frame,
# there is no LOCATION column. Instead, there is a name column with the full
# name of the country.

df_joined = df.join(df_oecd_locations)

In [11]:
df_joined

,LOCATION,SUBJECT,TIME,Value,location,location_full_name
0,AUS,INT_REC,2008,31159.8,AUS,Australia
1,AUS,INT_REC,2009,29980.7,AUT,Austria
2,AUS,INT_REC,2010,35165.5,BEL,Belgium
3,AUS,INT_REC,2011,38710.1,CAN,Canada
4,AUS,INT_REC,2012,38003.7,DNK,Denmark
...,...,...,...,...,...,...
1229,SRB,INT-EXP,2015,1253.644,<NA>,<NA>
1230,SRB,INT-EXP,2016,1351.098,<NA>,<NA>
1231,SRB,INT-EXP,2017,1549.183,<NA>,<NA>
1232,SRB,INT-EXP,2018,1837.317,<NA>,<NA>


In [12]:
df_joined.loc[df['SUBJECT'] == 'INT_REC'].groupby('location_full_name')['Value'].mean().sort_values().round(1).iloc[-5:]

location_full_name
Canada     38710.1
Hungary    39082.3
Italy      43959.1
Japan      47259.8
Korea      47930.9
Name: Value, dtype: double[pyarrow]

In [13]:
df_joined.loc[df['SUBJECT'] == 'INT-EXP'].groupby('location_full_name')['Value'].mean().sort_values().round(1).iloc[:5]

location_full_name
United States     25629.6
United Kingdom    27620.0
Brazil            31916.5
Israel            39381.5
Name: Value, dtype: double[pyarrow]

In [ ]:
# when using the full names for grouping, I did not get the same result, because there
# are abbreviations, e.g. "SRB" that do not have a corresponding entry in the oecd_locations file.